# Banjara AI Training (Roman Script)

This notebook trains an AI model to translate English to Banjara in **Roman script** (English alphabet).
We use Google's **FLAN-T5-Small**, a powerful and efficient model that handles Roman text perfectly.

In [ ]:
# 1. Install Dependencies
!pip install transformers datasets evaluate accelerate sentencepiece

In [ ]:
# 2. Unzip Dataset
# Upload your 'dataset.zip' to the Files section on the left before running this!
import os
if not os.path.exists('dataset'):
    !unzip dataset.zip

In [ ]:
# 3. Load Data
from datasets import load_dataset

# Point to the unzipped CSV files
dataset = load_dataset("csv", data_files={
    "train": "dataset/train/metadata.csv",
    "test": "dataset/test/metadata.csv"
})

print("Sample:", dataset['train'][0])

In [ ]:
# 4. Prepare Model (T5-Small)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model_checkpoint = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # T5 expects a task prefix
    inputs = ["translate English to Banjara: " + doc for doc in examples["english"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    
    labels = tokenizer(examples["banjara"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

In [ ]:
# 5. Train
training_args = Seq2SeqTrainingArguments(
    output_dir="./banjara_roman_model",
    eval_strategy="epoch",
    learning_rate=3e-4, # T5 uses higher LR
    per_device_train_batch_size=8, # T5-small fits easily
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=15, # More epochs for better adaptation
    predict_with_generate=True,
    fp16=False, # T5 sometimes unstable with fp16 on T4, using fp32 is safe and fast enough for small model
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# 6. Test Translation
from transformers import pipeline

# Save locally first
trainer.save_model("./banjara_roman_final")
tokenizer.save_pretrained("./banjara_roman_final")

translator = pipeline("translation", model="./banjara_roman_final", tokenizer="./banjara_roman_final")

test_sentences = [
    "Where are you going?",
    "This food is very tasty.",
    "My stomach is aching.",
    "Can you give me some water?",
    "I am going to the village tomorrow.",
]

print("Model Predictions (Roman Script):")
for sentence in test_sentences:
    # T5 needs the prefix during inference too
    input_text = "translate English to Banjara: " + sentence
    result = translator(input_text)[0]['translation_text']
    print(f"En: {sentence}")
    print(f"Bj: {result}")
    print("---")

In [ ]:
# 7. Save Model to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r ./banjara_roman_final /content/drive/MyDrive/banjara_roman_model